# Analisis dan Ekstraksi Fitur Time Series Kualitas Udara (NO2)
**Lokasi:** Kecamatan Kwanyar, Kabupaten Bangkalan
**Rentang Waktu:** 31 Agustus 2025 - 31 Agustus 2026

Tahap pertama dalam *pipeline* ini adalah mempersiapkan lingkungan kerja dengan menginstal library **TSFEL (Time Series Feature Extraction Library)** yang akan digunakan untuk mengekstrak puluhan fitur statistik dari data polusi.

In [ ]:
!pip install tsfel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 79.4 MB/s eta 0:00:00


### 1. Muat Data dan Pembersihan (Outlier & Missing Values)
Tahap ini bertujuan untuk membaca data historis `KualitasUdara_Kwanyar.csv`. Nilai ekstrem (outliers) akan dideteksi menggunakan metode IQR (Interquartile Range) lalu dihapus (diubah menjadi NaN). Setelah itu, seluruh *missing values* akan diisi kembali (imputasi) menggunakan metode interpolasi waktu.

In [ ]:
import pandas as pd
import numpy as np


df = pd.read_csv('KualitasUdara_Kwanyar.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

target_pollutant = 'NO2'

# Paksa kolom target jadi numerik, nilai yang gagal dikonversi -> NaN
df[target_pollutant] = pd.to_numeric(df[target_pollutant], errors='coerce')

n_missing_before = df[target_pollutant].isna().sum()
print(f"Jumlah nilai kosong bawaan/non-numerik: {n_missing_before}")

# Deteksi Outlier dengan IQR
Q1 = df[target_pollutant].quantile(0.25)
Q3 = df[target_pollutant].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Ubah nilai outlier menjadi NaN
df.loc[(df[target_pollutant] < lower_bound) | (df[target_pollutant] > upper_bound), target_pollutant] = np.nan

# Imputasi Missing Value & Outlier menggunakan interpolasi waktu
df_clean = df.set_index('date').interpolate(method='time').ffill().bfill()

print("Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi.")

# --- MENYIMPAN HASIL INTERPOLASI ---
df_clean_csv = df_clean.reset_index()
nama_file_bersih = 'KualitasUdara_Kwanyar_Cleaned.csv'
df_clean_csv.to_csv(nama_file_bersih, index=False)

print(f"File hasil interpolasi '{nama_file_bersih}' siap diunduh.")

Jumlah nilai kosong bawaan/non-numerik: 173
Pembersihan selesai! Outlier sudah dihilangkan dan Missing Value sudah diimputasi.
 File hasil interpolasi bernama 'KualitasUdara_Kwanyar_Cleaned.csv' 


### 2. Imputasi Missing Value dengan Interpolasi Waktu
Data yang hilang atau *outlier* yang telah dihapus tidak boleh dibiarkan kosong karena akan menggagalkan ekstraksi fitur TSFEL. Untuk mengatasinya, kita menggunakan metode **Interpolasi Waktu (Time Interpolation)**.

Metode ini memperkirakan nilai yang hilang dengan menarik garis lurus proporsional antara titik data sebelum dan sesudah nilai yang kosong.

**Rumus Dasar (Linear Interpolation):**
$$y = y_0 + (x - x_0) \frac{y_1 - y_0}{x_1 - x_0}$$

**Contoh Perhitungan Manual (Data Asli NO2 Kwanyar):**
Perekaman NO2 gagal (kosong) pada tanggal **10 September 2025**. Kita akan menghitung nilai penggantinya menggunakan data 9 Sept dan 11 Sept.

**Tabel Sebelum Imputasi:**
| Tanggal | Konsentrasi NO2 | Keterangan |
| :--- | :--- | :--- |
| 9 Sept 2025 | 0.000028185 | Data Valid |
| 10 Sept 2025 | NaN | **Missing Value** |
| 11 Sept 2025 | 0.000010713 | Data Valid |

**Perhitungan:**
$$y = 0.000028185 + (1) \frac{0.000010713 - 0.000028185}{2}$$
$$y = 0.000028185 - 0.000008736 = 0.000019449$$

Metode ini diimplementasikan secara otomatis menggunakan `df.interpolate(method='time')` untuk seluruh baris kosong.

### 3. Ekstraksi Fitur Polutan (TSFEL)
Tahap ini mengekstraksi data deret waktu polutan $NO_2$ yang sudah bersih menjadi 68 fitur statistik, spektral, dan temporal menggunakan library TSFEL. Data hasil ekstraksi ini kemudian akan diekspor untuk dibandingkan dengan daerah lain.

In [ ]:
import inspect
import tsfel.feature_extraction.features as tsfel_features

# Frekuensi sampling (1 per hari)
fs = 1
signal_1d = df_clean[target_pollutant].astype(float).values

# ---------- Daftar 68 fitur  ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang diminta:", len(FEATURE_LIST))

def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)

def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

row = {}
for fn_name in FEATURE_LIST:
    row[fn_name] = extract_one(fn_name, signal_1d, fs)

extracted_features_final = pd.DataFrame([row])

print(f"Berhasil! Jumlah fitur yang dihasilkan untuk {target_pollutant}: {extracted_features_final.shape[1]}")

# Menyimpan file ke dalam format CSV
nama_file_csv = f'{target_pollutant}_Kwanyar_TSFEL.csv'
extracted_features_final.to_csv(nama_file_csv, index=False)
print(f"File ekstraksi {nama_file_csv} sudah tersimpan.")

Jumlah fitur yang diminta: 68
Berhasil! Jumlah fitur yang dihasilkan untuk NO2: 68
File ekstraksi NO2_Kwanyar_TSFEL.csv sudah tersimpan.


### Penjelasan 68 Fitur Ekstraksi

### **1. Statistical Domain (21 Fitur)**

1.  **abs_energy**: Total kuadrat dari seluruh nilai polusi.
   
    $$E = \sum_{i=1}^{N} x_i^2$$
    
    *   $E$: Energi absolut (Absolute Energy)
    *   $N$: Jumlah total baris data pengamatan
    *   $x_i$: Nilai kadar polusi pada waktu ke-$i$

2.  **average_power**: Rata-rata daya (energi per unit waktu) dari sinyal.
    $$P_{avg} = \frac{1}{N} \sum_{i=1}^{N} x_i^2$$
    *   $P_{avg}$: Rata-rata daya polusi
    *   $N$: Jumlah total baris data
    *   $x_i$: Nilai kadar polusi pada waktu ke-$i$

3.  **calc_max**: Nilai polusi paling tinggi dalam dataset.
    $$Max = \max(x_1, x_2, \dots, x_N)$$
    *   $Max$: Nilai puncak tertinggi (maksimum)
    *   $x$: Deret data polusi

4.  **calc_mean**: Nilai rata-rata dari seluruh deret waktu.
    $$\mu = \frac{1}{N} \sum_{i=1}^{N} x_i$$
    *   $\mu$: Nilai rata-rata (mean)
    *   $N$: Jumlah total baris data
    *   $x_i$: Nilai kadar polusi pada waktu ke-$i$

5.  **calc_median**: Nilai tengah data saat diurutkan dari terkecil hingga terbesar.
    $$Median = x_{(N+1)/2}$$
    *   $Median$: Nilai tengah kumpulan data
    *   $x$: Deret data polusi yang sudah diurutkan
    *   $N$: Jumlah total baris data (contoh rumus untuk $N$ ganjil)

6.  **calc_min**: Nilai polusi paling rendah dalam dataset.
    $$Min = \min(x_1, x_2, \dots, x_N)$$
    *   $Min$: Nilai batas bawah (minimum)
    *   $x$: Deret data polusi

7.  **calc_std**: Standar deviasi (penyebaran data dari rata-ratanya).
    $$\sigma = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N} (x_i - \mu)^2}$$
    *   $\sigma$: Standar deviasi
    *   $N$: Jumlah total baris data
    *   $x_i$: Nilai kadar polusi pada waktu ke-$i$
    *   $\mu$: Nilai rata-rata polusi

8.  **calc_var**: Varians (kuadrat dari standar deviasi).
    $$\sigma^2 = \frac{1}{N-1} \sum_{i=1}^{N} (x_i - \mu)^2$$
    *   $\sigma^2$: Varians
    *   $N$: Jumlah total baris data
    *   $x_i$: Nilai kadar polusi pada waktu ke-$i$
    *   $\mu$: Nilai rata-rata polusi

9.  **ecdf**: Fungsi Distribusi Kumulatif Empiris. Menghitung proporsi data yang bernilai kurang dari atau sama dengan ambang batas.
    $$F(t) = \frac{1}{N} \sum_{i=1}^{N} I(x_i \le t)$$
    *   $F(t)$: Probabilitas kumulatif pada ambang batas $t$
    *   $N$: Jumlah total data
    *   $I$: Fungsi indikator (bernilai 1 jika $x_i \le t$, dan 0 jika sebaliknya)
    *   $t$: Nilai ambang batas yang diuji

10. **ecdf_percentile**: Menemukan nilai polusi pada batas proporsi kumulatif tertentu (misal persentil ke-20).
    $$P_p = \inf \{t : F(t) \ge p\}$$
    *   $P_p$: Nilai polusi pada persentil $p$
    *   $F(t)$: Fungsi distribusi kumulatif ECDF
    *   $p$: Persentase probabilitas sasaran (contoh: 0.20 untuk 20%)

11. **ecdf_percentile_count**: Jumlah baris data yang nilainya jatuh di antara dua batas persentil.
    $$Count = \sum_{i=1}^{N} I(P_{bawah} \le x_i \le P_{atas})$$
    *   $Count$: Jumlah baris data yang memenuhi syarat
    *   $I$: Fungsi indikator
    *   $P_{bawah}$: Nilai polusi di batas persentil bawah
    *   $P_{atas}$: Nilai polusi di batas persentil atas

12. **ecdf_slope**: Kemiringan garis antara dua batas persentil kurva ECDF.
    $$Slope = \frac{P_{atas} - P_{bawah}}{p_{atas} - p_{bawah}}$$
    *   $Slope$: Tingkat kemiringan ECDF
    *   $P_{atas}, P_{bawah}$: Nilai polusi pada batas atas dan bawah
    *   $p_{atas}, p_{bawah}$: Probabilitas persentil atas dan bawah (contoh: 0.80 dan 0.20)

13. **entropy**: Entropi Shannon dari distribusi frekuensi nilai polusi.
    $$H_{shannon} = -\sum_{i=1}^{K} p(x_i) \log_2 p(x_i)$$
    *   $H_{shannon}$: Nilai entropi (keragaman informasi)
    *   $K$: Jumlah kelompok/bin nilai polusi yang unik
    *   $p(x_i)$: Probabilitas (peluang) munculnya nilai $x_i$ dalam dataset

14. **hist_mode**: Nilai yang paling sering muncul (modus), didapat dari bin frekuensi tertinggi dalam histogram.
    $$Mode = \arg\max_{x} (f(x))$$
    *   $Mode$: Nilai paling dominan (modus)
    *   $f(x)$: Jumlah frekuensi kemunculan untuk rentang nilai $x$

15. **interq_range**: Jarak interkuartil (selisih kuartil 3 dan kuartil 1).
    $$IQR = Q_3 - Q_1$$
    *   $IQR$: Interquartile Range
    *   $Q_3$: Kuartil ketiga (persentil 75%)
    *   $Q_1$: Kuartil pertama (persentil 25%)

16. **kurtosis**: Derajat keruncingan (banyaknya ekor tebal/outlier) dari sebaran data.
    $$Kurtosis = \frac{1}{N} \sum_{i=1}^{N} \left(\frac{x_i - \mu}{\sigma}\right)^4$$
    *   $Kurtosis$: Nilai keruncingan
    *   $N$: Jumlah total baris data
    *   $x_i$: Nilai kadar polusi
    *   $\mu$: Nilai rata-rata polusi
    *   $\sigma$: Standar deviasi

17. **mean_abs_deviation**: Rata-rata dari nilai mutlak selisih setiap data dengan rata-ratanya.
    $$MAD_{mean} = \frac{1}{N} \sum_{i=1}^{N} |x_i - \mu|$$
    *   $MAD_{mean}$: Mean Absolute Deviation
    *   $N$: Jumlah total data
    *   $x_i$: Nilai kadar polusi
    *   $\mu$: Nilai rata-rata polusi

18. **median_abs_deviation**: Nilai tengah (median) dari mutlak selisih setiap data dengan median datanya.
    $$MAD_{median} = \text{median}(|x_i - Median|)$$
    *   $MAD_{median}$: Median Absolute Deviation
    *   $x_i$: Nilai kadar polusi
    *   $Median$: Nilai tengah dari keseluruhan data

19. **pk_pk_distance**: Jarak dari puncak ke puncak ekstrem (rentang absolut maksimum).
    $$Distance_{pk} = \max(x) - \min(x)$$
    *   $Distance_{pk}$: Jarak Peak-to-Peak
    *   $\max(x)$: Nilai polusi tertinggi
    *   $\min(x)$: Nilai polusi terendah

20. **rms**: Akar dari nilai rata-rata kuadrat (Root Mean Square).
    $$RMS = \sqrt{\frac{1}{N} \sum_{i=1}^{N} x_i^2}$$
    *   $RMS$: Root Mean Square
    *   $N$: Jumlah total data
    *   $x_i$: Nilai kadar polusi

21. **skewness**: Derajat kemencengan (ketidaksimetrisan) distribusi data polusi.
    $$Skewness = \frac{1}{N} \sum_{i=1}^{N} \left(\frac{x_i - \mu}{\sigma}\right)^3$$
    *   $Skewness$: Nilai kemencengan
    *   $N$: Jumlah total baris data
    *   $x_i$: Nilai kadar polusi
    *   $\mu$: Rata-rata polusi
    *   $\sigma$: Standar deviasi

---

### **2. Temporal Domain (15 Fitur)**

22. **auc**: Total luas area pergerakan data di bawah kurva (integrasi trapesium).
    $$AUC = \sum_{i=1}^{N-1} \frac{(x_i + x_{i+1})}{2}$$
    *   $AUC$: Area Under Curve
    *   $N$: Jumlah titik waktu pengamatan
    *   $x_i$: Nilai polusi pada waktu ke-$i$
    *   $x_{i+1}$: Nilai polusi pada waktu berikutnya

23. **autocorr**: Autokorelasi, korelasi sinyal terhadap dirinya sendiri pada jeda waktu (lag) tertunda.
    $$R(k) = \frac{\sum_{i=1}^{N-k} (x_i - \mu)(x_{i+k} - \mu)}{\sum_{i=1}^{N} (x_i - \mu)^2}$$
    *   $R(k)$: Koefisien autokorelasi
    *   $k$: Jumlah jeda waktu (lag)
    *   $N$: Jumlah total data
    *   $x_i$: Nilai polusi pada waktu ke-$i$
    *   $\mu$: Rata-rata polusi keseluruhan

24. **calc_centroid**: Pusat massa waktu dari kurva polusi. Mengukur pada waktu ke-berapa polusi paling terkonsentrasi.
    $$t_c = \frac{\sum_{i=1}^{N} i \cdot x_i}{\sum_{i=1}^{N} x_i}$$
    *   $t_c$: Temporal centroid (pusat massa waktu)
    *   $i$: Indeks urutan waktu
    *   $x_i$: Nilai polusi pada indeks ke-$i$

25. **distance**: Total perpindahan absolut yang dilewati oleh sinyal dari awal hingga akhir.
    $$D_{seq} = \sum_{i=1}^{N-1} |x_{i+1} - x_i|$$
    *   $D_{seq}$: Jarak lintasan sekuensial
    *   $x_{i+1} - x_i$: Selisih lonjakan polusi antar waktu

26. **lempel_ziv**: Nilai kompleksitas menggunakan algoritma kompresi Lempel-Ziv.
    $$LZ = c(N) \left( \frac{\log_2 N}{N} \right)$$
    *   $LZ$: Kompleksitas Lempel-Ziv
    *   $c(N)$: Jumlah pola urutan biner unik yang ditemukan dalam data
    *   $N$: Total panjang data

27. **mean_abs_diff**: Rata-rata dari nilai absolut lonjakan antar waktu.
    $$MAD_{diff} = \frac{1}{N-1} \sum_{i=1}^{N-1} |x_{i+1} - x_i|$$
    *   $MAD_{diff}$: Mean Absolute Difference
    *   $N$: Jumlah titik waktu
    *   $x_{i+1} - x_i$: Perbedaan nilai berurutan

28. **mean_diff**: Rata-rata murni dari selisih antar data (termasuk nilai minus).
    $$M_{diff} = \frac{1}{N-1} \sum_{i=1}^{N-1} (x_{i+1} - x_i)$$
    *   $M_{diff}$: Mean Difference
    *   $x_{i+1} - x_i$: Perbedaan mentah nilai polusi berurutan

29. **median_abs_diff**: Nilai tengah (median) dari lonjakan absolut antar waktu.
    $$MedAD_{diff} = \text{median}(|x_{i+1} - x_i|)$$
    *   $MedAD_{diff}$: Median Absolute Difference
    *   $|x_{i+1} - x_i|$: Kumpulan nilai absolut selisih berurutan

30. **median_diff**: Nilai tengah dari lonjakan mentah berurutan.
    $$Med_{diff} = \text{median}(x_{i+1} - x_i)$$
    *   $Med_{diff}$: Median Difference

31. **negative_turning**: Jumlah kali kurva polusi membentuk lembah (lokal minimum).
    $$Turns_{-} = \sum_{i=2}^{N-1} I(x_{i-1} > x_i < x_{i+1})$$
    *   $Turns_{-}$: Jumlah titik balik negatif
    *   $I$: Fungsi indikator yang bernilai 1 jika $x_i$ lebih kecil dari data sebelum dan sesudahnya

32. **neighbourhood_peaks**: Jumlah puncak dominan relatif terhadap data di sekitarnya.
    $$Peaks = \sum_{i} I(\text{is\_peak}(x_i, threshold))$$
    *   $Peaks$: Jumlah puncak lokal yang valid
    *   $threshold$: Ambang batas minimum kelonjakan puncak

33. **positive_turning**: Jumlah kali kurva polusi membentuk gunung (lokal maksimum).
    $$Turns_{+} = \sum_{i=2}^{N-1} I(x_{i-1} < x_i > x_{i+1})$$
    *   $Turns_{+}$: Jumlah titik balik positif
    *   $I$: Fungsi indikator yang bernilai 1 jika $x_i$ lebih besar dari data sebelum dan sesudahnya

34. **slope**: Nilai kemiringan ($m$) dari regresi linier keseluruhan tren waktu polusi.
    $$Slope = \frac{N \sum (i \cdot x_i) - \sum i \sum x_i}{N \sum i^2 - (\sum i)^2}$$
    *   $Slope$: Kemiringan tren waktu
    *   $N$: Total titik pengamatan
    *   $i$: Indeks waktu (variabel X pada regresi)
    *   $x_i$: Nilai polusi (variabel Y pada regresi)

35. **sum_abs_diff**: Total kumulatif dari seluruh selisih absolut lonjakan waktu ke waktu.
    $$SumDiff = \sum_{i=1}^{N-1} |x_{i+1} - x_i|$$
    *   $SumDiff$: Jumlah fluktuasi total
    *   $|x_{i+1} - x_i|$: Selisih absolut data berurutan

36. **zero_cross**: Jumlah titik waktu di mana polusi memotong rata-ratanya atau memotong nilai nol.
    $$ZC = \frac{1}{2} \sum_{i=1}^{N-1} |\text{sgn}(x_{i+1}) - \text{sgn}(x_i)|$$
    *   $ZC$: Zero Crossing Rate
    *   $\text{sgn}$: Fungsi tanda (menghasilkan 1 jika positif, -1 jika negatif)
    *   $x_i$: Nilai polusi yang sudah dikurangi rata-ratanya

---

### **3. Spectral Domain (26 Fitur)**

37. **fundamental_frequency**: Frekuensi dasar terendah yang memiliki magnitudo paling dominan.
    $$f_{fund} = \arg\max_{k>0} |X(k)|$$
    *   $f_{fund}$: Frekuensi fundamental
    *   $X(k)$: Magnitudo spektrum (hasil FFT) pada indeks frekuensi ke-$k$

38. **human_range_energy**: Energi pada rentang pita frekuensi biologis tertentu.
    $$E_{human} = \sum_{f(k) \in [0.6, 2.5]} |X(k)|^2$$
    *   $E_{human}$: Total energi rentang manusia
    *   $f(k)$: Nilai frekuensi pada indeks ke-$k$
    *   $|X(k)|^2$: Daya (power) pada frekuensi tersebut

39. **lpcc**: *Linear Prediction Cepstral Coefficients*. Hasil ekstraksi amplop spektrum daya menggunakan filter prediktif linear.
    $$C_{lpcc} = \text{Cepstrum}(\text{LPC}(X))$$
    *   $C_{lpcc}$: Koefisien LPCC
    *   $LPC(X)$: Pemodelan filter Linear Predictive Coding pada deret waktu $X$

40. **max_frequency**: Indeks frekuensi yang menyimpan magnitudo energi paling tinggi.
    $$f_{max} = f(\arg\max_{k} |X(k)|^2)$$
    *   $f_{max}$: Frekuensi puncak maksimum
    *   $|X(k)|^2$: Daya spektral di indeks ke-$k$

41. **max_power_spectrum**: Besaran nilai daya energi tertinggi itu sendiri.
    $$P_{max} = \max_k (|X(k)|^2)$$
    *   $P_{max}$: Nilai daya spektral maksimum
    *   $|X(k)|^2$: Energi daya spektral

42. **median_frequency**: Frekuensi tengah yang membagi energi spektrum menjadi dua bagian sama besar.
    $$f_{med} : \sum_{k=1}^{med} |X(k)|^2 = \frac{1}{2} \sum_{k=1}^{K} |X(k)|^2$$
    *   $f_{med}$: Median Frequency
    *   $K$: Jumlah total *bin* frekuensi

43. **mfcc**: *Mel-Frequency Cepstral Coefficients*. Koefisien spektral dalam skala logaritmik Mel.
    $$C_m = \sum_{k=1}^{K} (\log S_k) \cos \left[ m \left( k - \frac{1}{2} \right) \frac{\pi}{K} \right]$$
    *   $C_m$: Koefisien MFCC ke-$m$
    *   $S_k$: Energi keluaran dari *filterbank* skala-Mel ke-$k$
    *   $K$: Total *filterbank*

44. **power_bandwidth**: Lebar pita frekuensi (Hz) yang menampung daya spektral utama.
    $$Band_{width} = f_{atas} - f_{bawah}$$
    *   $Band_{width}$: Lebar pita frekuensi
    *   $f_{atas}, f_{bawah}$: Titik frekuensi atas dan bawah yang membatasi 99% energi

45. **spectral_centroid**: Titik berat rata-rata frekuensi (Pusat massa spektrum).
    $$Centroid = \frac{\sum_{k} f(k)|X(k)|^2}{\sum_{k} |X(k)|^2}$$
    *   $Centroid$: Pusat massa spektral (dalam satuan Hz)
    *   $f(k)$: Nilai frekuensi
    *   $|X(k)|^2$: Daya (bobot) pada frekuensi tersebut

46. **spectral_decrease**: Tingkat kelandaian spektrum menuju frekuensi tinggi.
    $$Decrease = \frac{1}{\sum_{k=2}^{K} |X(k)|} \sum_{k=2}^{K} \frac{|X(k)| - |X(1)|}{k-1}$$
    *   $Decrease$: Indeks penurunan spektral
    *   $|X(1)|$: Magnitudo spektrum di frekuensi dasar terendah
    *   $|X(k)|$: Magnitudo di indeks ke-$k$

47. **spectral_distance**: Jarak Euclidean antar bin spektrum daya berurutan.
    $$D_{spec} = \sqrt{\sum_{k} (|X_1(k)| - |X_2(k)|)^2}$$
    *   $D_{spec}$: Spectral Distance
    *   $X_1, X_2$: Spektrum frekuensi dari dua jendela waktu yang berbeda

48. **spectral_entropy**: Entropi kerumitan sinyal frekuensi.
    $$H_{spec} = -\sum_{k} p_k \log_2 p_k \quad \text{dimana} \quad p_k = \frac{|X(k)|^2}{\sum |X(k)|^2}$$
    *   $H_{spec}$: Spectral Entropy
    *   $p_k$: Rasio probabilitas (bobot energi) pada indeks frekuensi ke-$k$

49. **spectral_kurtosis**: Derajat keruncingan kurva spektrum frekuensi (mengukur adanya lonjakan *outlier* spektral).
    $$Kurt_{spec} = \frac{\sum_{k} (f(k) - Centroid)^4 |X(k)|^2}{\left(\sum_{k} (f(k) - Centroid)^2 |X(k)|^2\right)^2}$$
    *   $Kurt_{spec}$: Spectral Kurtosis
    *   $f(k)$: Frekuensi
    *   $Centroid$: Titik pusat spektrum

50. **spectral_positive_turning**: Jumlah puncak lokal (gunung) di dalam sebaran domain frekuensi.
    $$Turns_{spec+} = \sum_{k=2}^{K-1} I(|X(k-1)| < |X(k)| > |X(k+1)|)$$
    *   $Turns_{spec+}$: Jumlah puncak spektral
    *   $|X(k)|$: Magnitudo energi di bin ke-$k$

51. **spectral_roll_off**: Titik frekuensi batas di mana 85% energi berada di bawahnya.
    $$f_c : \sum_{k=1}^{c} |X(k)|^2 = 0.85 \sum_{k=1}^{K} |X(k)|^2$$
    *   $f_c$: Frekuensi Roll-Off
    *   $K$: Total bin frekuensi

52. **spectral_roll_on**: Titik frekuensi terendah di mana 15% energi berada di bawahnya (batas bawah pendar energi).
    $$f_{on} : \sum_{k=1}^{on} |X(k)|^2 = 0.15 \sum_{k=1}^{K} |X(k)|^2$$
    *   $f_{on}$: Frekuensi Roll-On

53. **spectral_skewness**: Derajat kemencengan kurva sebaran magnitudo spektrum.
    $$Skew_{spec} = \frac{\sum_{k} (f(k) - Centroid)^3 |X(k)|^2}{\left(\sum_{k} (f(k) - Centroid)^2 |X(k)|^2\right)^{1.5}}$$
    *   $Skew_{spec}$: Spectral Skewness
    *   $Centroid$: Pusat spektrum rata-rata

54. **spectral_slope**: Kemiringan garis regresi dari pola penyebaran spektrum secara keseluruhan.
    $$Slope_{spec} = \frac{K \sum (f(k) \cdot |X(k)|) - \sum f(k) \sum |X(k)|}{K \sum f(k)^2 - (\sum f(k))^2}$$
    *   $Slope_{spec}$: Kemiringan spektrum (mewakili apakah sinyal didominasi frekuensi rendah atau tinggi)

55. **spectral_spread**: Standar deviasi (lebar sebaran energi) di sekitar *Spectral Centroid*.
    $$Spread = \sqrt{\frac{\sum_k (f(k) - Centroid)^2 |X(k)|^2}{\sum_k |X(k)|^2}}$$
    *   $Spread$: Persebaran spektral
    *   $Centroid$: Titik berat frekuensi

56. **spectral_variation**: Jumlah fluktuasi perbedaan wujud spektrum dari waktu ke waktu.
    $$Var_{spec} = \sum_{k} ||X_t(k)| - |X_{t-1}(k)||$$
    *   $Var_{spec}$: Variasi spektral
    *   $X_t$: Spektrum frekuensi di waktu saat ini
    *   $X_{t-1}$: Spektrum frekuensi di jendela waktu sebelumnya

57. **spectrogram_mean_coeff**: Nilai rata-rata dari seluruh sel matriks STFT (*Short-Time Fourier Transform*).
    $$Mean_{STFT} = \frac{1}{T \cdot K} \sum_{t=1}^{T} \sum_{k=1}^{K} |S(t, k)|$$
    *   $Mean_{STFT}$: Rata-rata koefisien spektrogram
    *   $T$: Total blok/jendela waktu
    *   $K$: Total bin frekuensi
    *   $S(t, k)$: Koefisien STFT pada waktu $t$ frekuensi $k$

58. **wavelet_abs_mean**: Rata-rata dari nilai absolut koefisien Transformasi Wavelet Diskrit.
    $$W_{abs} = \frac{1}{J} \sum_{j=1}^{J} |C_j|$$
    *   $W_{abs}$: Rata-rata mutlak wavelet
    *   $J$: Jumlah total koefisien wavelet yang dihasilkan
    *   $C_j$: Nilai koefisien wavelet ke-$j$

59. **wavelet_energy**: Total energi dari kumpulan koefisien wavelet.
    $$W_{energy} = \sum_{j=1}^{J} C_j^2$$
    *   $W_{energy}$: Energi total wavelet
    *   $C_j$: Koefisien wavelet

60. **wavelet_entropy**: Entropi kerumitan informasi di dalam set koefisien wavelet.
    $$H_{wavelet} = -\sum_{j=1}^{J} p_j \log_2 p_j$$
    *   $H_{wavelet}$: Entropi wavelet
    *   $p_j$: Energi koefisien yang dinormalisasi ($C_j^2 / \sum C_i^2$)

61. **wavelet_std**: Standar deviasi dari sebaran fluktuasi koefisien wavelet.
    $$W_{std} = \sqrt{\frac{1}{J-1} \sum_{j=1}^{J} (C_j - \mu_w)^2}$$
    *   $W_{std}$: Standar deviasi wavelet
    *   $\mu_w$: Nilai rata-rata dari seluruh koefisien wavelet

62. **wavelet_var**: Varians dari penyebaran koefisien wavelet.
    $$W_{var} = \frac{1}{J-1} \sum_{j=1}^{J} (C_j - \mu_w)^2$$
    *   $W_{var}$: Varians wavelet
    *   $\mu_w$: Rata-rata koefisien wavelet

---

### **4. Fractal Domain (6 Fitur)**

63. **dfa (Detrended Fluctuation Analysis)**: Eksponen fraktal berdasarkan fungsi fluktuasi $F(n)$ yang sudah dihapus tren lokalnya ($y_n(k)$).
    $$F(n) = \sqrt{\frac{1}{N} \sum_{k=1}^{N} [y(k) - y_n(k)]^2} \propto n^\alpha$$
    *   $F(n)$: Fungsi fluktuasi rata-rata akar kuadrat
    *   $y(k)$: Deret kumulatif dari sinyal polusi
    *   $y_n(k)$: Garis regresi tren lokal pada jendela skala $n$
    *   $\alpha$: Nilai DFA (eksponen yang dihasilkan)

64. **higuchi_fractal_dimension**: Menghitung dimensi fraktal $D$ dari perkiraan panjang kurva deret waktu polusi $L(k)$ di berbagai interval waktu $k$.
    $$L(k) \propto k^{-D}$$
    *   $L(k)$: Panjang geometris kurva deret waktu
    *   $k$: Jeda interval waktu observasi
    *   $D$: Nilai Dimensi Fraktal Higuchi

65. **hurst_exponent**: Mengukur sifat memori jangka panjang polusi (apakah polusi terus naik, atau berbalik turun).
    $$\mathbb{E}\left[\frac{R(n)}{S(n)}\right] = C n^H$$
    *   $R(n)$: Rentang jangkauan (nilai maksimal dikurangi minimal) data kumulatif
    *   $S(n)$: Standar deviasi data dalam skala jendela pengamatan $n$
    *   $n$: Lebar rentang waktu pengamatan
    *   $H$: Hurst Exponent

66. **maximum_fractal_length**: Menghitung batas ukuran segmen skala log-log plot maksimal sebelum data tersebut kehilangan karakterisik fraktalnya.
    $$MFL = \arg\max_{n} (\text{Linearitas Log-Log Plot } L(n))$$
    *   $MFL$: Maximum Fractal Length
    *   $n$: Skala interval pengamatan

67. **mse (Multiscale Entropy)**: Entropi Sampel ($SampEn$) yang dihitung pada deret waktu yang sudah diturunkan resolusinya (dirata-ratakan) dalam skala $\tau$.
    $$SampEn(y^{(\tau)}) = -\ln \frac{A}{B}$$
    *   $SampEn$: Sample Entropy (entropi sampel)
    *   $y^{(\tau)}$: Sinyal yang telah diskalakan dengan merata-ratakan setiap titik sejauh skala $\tau$
    *   $A, B$: Jumlah pasangan *template* sinyal yang masih mirip setelah diperpanjang satu titik observasi

68. **petrosian_fractal_dimension**: Estimasi dimensi fraktal berbasis jumlah pembalikan arah ($N_{\Delta}$) sinyal polusi.
    $$PFD = \frac{\log_{10}(N)}{\log_{10}(N) + \log_{10}\left(\frac{N}{N + 0.4N_{\Delta}}\right)}$$
    *   $PFD$: Petrosian Fractal Dimension
    *   $N$: Total baris data deret waktu
    *   $N_{\Delta}$: Jumlah perubahan tanda (naik $\rightarrow$ turun, atau turun $\rightarrow$ naik) dari turunan nilai polusi